In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, roc_curve, auc, confusion_matrix
)
from sklearn.utils.class_weight import compute_sample_weight
from statsmodels.stats.outliers_influence import variance_inflation_factor

sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'font.family': 'sans-serif',
    'axes.spines.top': False,
    'axes.spines.right': False,
})
FIGURES = Path('../outputs/figures')
FIGURES.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42

## Step 1 — Target Variable

The dropout prediction model treats children as **positive cases (1) if they have dropped out** and negative cases (0) if they are currently enrolled. Children who were *never enrolled* are excluded — they represent a distinct phenomenon (non-entry) with different causal drivers than mid-enrolment dropout.

In [ ]:
LOAD_COLS = [
    'age', 'gender', 'grade', 'has_electricity', 'has_tv',
    'has_smartphone', 'has_internet', 'has_computer', 'has_toilet',
    'earning_members', 'receives_bisp', 'receives_ehsaas', 'flood_affected',
    'education_expense', 'enrollment_status', 'province',
]
raw  = pd.read_csv('../Data:/child_merged.csv', usecols=LOAD_COLS)
base = raw[raw['enrollment_status'].isin(['Enrolled', 'Dropped out'])].copy()
base['target'] = (base['enrollment_status'] == 'Dropped out').astype(int)

print('── STEP 1: Target variable ──')
print(f'Total children (Enrolled + Dropped out): {len(base):,}')
vc = base['target'].value_counts().rename({0: 'Enrolled', 1: 'Dropped out'})
print(pd.DataFrame({'count': vc, '%': (vc / len(base) * 100).round(1)}))

## Step 2 — Feature Engineering

### Why age and grade are excluded as continuous predictors

Previous versions of this notebook used raw `age` and `grade` as continuous features. Their Pearson correlation is r = 0.907 — near-perfect collinearity. When both enter a logistic regression, the model assigns arbitrarily large and opposite coefficients (±9–12 in standardised form) to satisfy the constraint that their *sum* represents the age-grade relationship. These coefficients look decisive but are statistical noise: change the regularisation strength by a small amount and the individual values flip sign entirely. They also dominate the provincial coefficient chart, making all other predictors invisible.

The fix is to drop both and replace grade with a **three-level categorical variable `school_stage`**:

| Level | Grades | Reference? |
|---|---|---|
| `primary` | 1 – 5 | Yes (dropped, baseline) |
| `middle` | 6 – 8 | No — `school_stage_middle` dummy |
| `secondary` | 9 – 10 | No — `school_stage_secondary` dummy |

This captures *where in the system* dropout happens (primary vs middle vs secondary transition) without continuous collinearity. Age is excluded entirely — it is made redundant once school stage is known.

### Data leakage notes
- **`school_type`** and **`paid_tuition`** remain excluded: both are `NaN` for all dropped-out children, making them perfect leaks.

### Imputation strategy
Household asset flags and welfare receipt: `NaN → 0`. `education_expense`: `NaN → 0` (dropped-out children have zero current school expense). `earning_members`: `NaN → median`. Rows are dropped only if `grade` or `gender` is missing.

In [ ]:
df = base.copy()

FILL_ZERO = [
    'has_electricity', 'has_tv', 'has_smartphone', 'has_internet',
    'has_computer', 'has_toilet', 'receives_bisp', 'receives_ehsaas',
    'flood_affected', 'education_expense',
]
for col in FILL_ZERO:
    df[col] = df[col].fillna(0)
df['earning_members'] = df['earning_members'].fillna(df['earning_members'].median())

df = df.dropna(subset=['grade', 'gender'])
df['grade'] = df['grade'].astype(int)

# school_stage: primary = grades 1-5, middle = 6-8, secondary = 9-10
df['school_stage'] = pd.cut(
    df['grade'], bins=[0, 5, 8, 10],
    labels=['primary', 'middle', 'secondary']
)
df = df.dropna(subset=['school_stage'])  # removes grade 0, 11-14

df = pd.get_dummies(df, columns=['school_stage', 'gender'], drop_first=False, dtype=float)
df = df.drop(columns=['school_stage_primary', 'gender_Female', 'gender_Transgender'],
             errors='ignore')

FEATURE_COLS = [
    'school_stage_middle', 'school_stage_secondary',
    'earning_members', 'education_expense',
    'has_electricity', 'has_tv', 'has_smartphone', 'has_internet', 'has_computer',
    'has_toilet', 'receives_bisp', 'receives_ehsaas', 'flood_affected', 'gender_Male',
]
FEATURE_COLS = [c for c in FEATURE_COLS if c in df.columns]

# Pretty display names (used in all charts)
PRETTY = {
    'school_stage_middle':    'Stage: Middle (gr. 6–8)',
    'school_stage_secondary': 'Stage: Secondary (gr. 9–10)',
    'earning_members':        'Earning members',
    'education_expense':      'Education expense',
    'has_electricity':        'Has electricity',
    'has_tv':                 'Has TV',
    'has_smartphone':         'Has smartphone',
    'has_internet':           'Has internet',
    'has_computer':           'Has computer',
    'has_toilet':             'Has toilet',
    'receives_bisp':          'Receives BISP',
    'receives_ehsaas':        'Receives Ehsaas',
    'flood_affected':         'Flood affected',
    'gender_Male':            'Gender: Male',
}

print('── STEP 2: Feature engineering ──')
print(f'Final sample: {len(df):,} children')
vc2 = df['target'].value_counts().rename({0: 'Enrolled', 1: 'Dropped out'})
print(pd.DataFrame({'count': vc2, '%': (vc2 / len(df) * 100).round(1)}))
print(f'\nFeatures ({len(FEATURE_COLS)}): {FEATURE_COLS}')

# School-stage breakdown
stage_tbl = df.groupby(
    pd.cut(df['grade'], bins=[0,5,8,10], labels=['Primary','Middle','Secondary'])
).agg(n=('target','count'), dropouts=('target','sum'))
stage_tbl['dropout_rate'] = (stage_tbl['dropouts'] / stage_tbl['n'] * 100).round(1)
print('\nDropout rate by school stage:')
print(stage_tbl)

## Step 3 — Model Training

**Class imbalance**: 2.7% of children in the analytic sample have dropped out. Balanced class weights are applied to both models so dropout cases are penalised more heavily than the majority-class enrolled cases.

**Primary metric**: ROC-AUC. This measures the model's ability to *rank* children by dropout risk regardless of the decision threshold — directly useful for targeting a fixed-capacity intervention (e.g., welfare visits to the top 10% of predicted-risk children) at the highest-risk group.

In [ ]:
X = df[FEATURE_COLS].astype(float)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# Logistic Regression
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE)
lr.fit(X_train_sc, y_train)

# Gradient Boosting
sw_train = compute_sample_weight('balanced', y_train)
gbc = GradientBoostingClassifier(n_estimators=200, max_depth=4, random_state=RANDOM_STATE)
gbc.fit(X_train, y_train, sample_weight=sw_train)

def eval_model(name, model, X_ev, y_ev):
    y_pred = model.predict(X_ev)
    y_prob = model.predict_proba(X_ev)[:, 1]
    fpr, tpr, _ = roc_curve(y_ev, y_prob)
    roc_auc = auc(fpr, tpr)
    rpt = classification_report(y_ev, y_pred,
                                target_names=['Enrolled', 'Dropped out'],
                                output_dict=True)
    do = rpt['Dropped out']
    print(f'\n── {name} ──')
    print(f'  ROC-AUC  : {roc_auc:.4f}')
    print(f'  Accuracy : {rpt["accuracy"]:.4f}')
    print(f'  Precision (dropout): {do["precision"]:.4f}')
    print(f'  Recall    (dropout): {do["recall"]:.4f}')
    print(f'  F1        (dropout): {do["f1-score"]:.4f}')
    return fpr, tpr, roc_auc

lr_fpr,  lr_tpr,  lr_auc  = eval_model('Logistic Regression', lr,  X_test_sc, y_test)
gbc_fpr, gbc_tpr, gbc_auc = eval_model('Gradient Boosting',   gbc, X_test,    y_test)

## Step 4 — Visualisations

In [ ]:
# ── 4a: ROC Curves ──
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(lr_fpr,  lr_tpr,  color='#1f77b4', lw=2,
        label=f'Logistic Regression  (AUC = {lr_auc:.3f})')
ax.plot(gbc_fpr, gbc_tpr, color='#d62728', lw=2,
        label=f'Gradient Boosting    (AUC = {gbc_auc:.3f})')
ax.plot([0, 1], [0, 1], color='#999999', lw=1, linestyle='--', label='Random classifier')
ax.set_xlabel('False Positive Rate', labelpad=8)
ax.set_ylabel('True Positive Rate', labelpad=8)
ax.set_title('ROC Curves — Dropout Prediction Models', fontsize=13, fontweight='bold', pad=12)
ax.legend(frameon=False, fontsize=10, loc='lower right')
sns.despine(ax=ax)
plt.tight_layout()
fig.savefig(FIGURES / '07a_roc_curves.png', bbox_inches='tight')
fig.savefig(FIGURES / '07a_roc_curves.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ── 4b: Feature Importances (GBC) ──
fi = pd.Series(gbc.feature_importances_, index=FEATURE_COLS)
fi.index = [PRETTY.get(f, f) for f in fi.index]
fi = fi.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
colors = ['#4c72b0' if v < fi.median() else '#d62728' for v in fi.values]
ax.barh(fi.index, fi.values, color=colors, zorder=3)
for i, (feat, val) in enumerate(fi.items()):
    ax.text(val + 0.0005, i, f'{val:.3f}', va='center', fontsize=8.5, color='#333333')
ax.set_xlabel('Feature importance (mean decrease in impurity)', labelpad=8)
ax.set_title('Feature Importances — Gradient Boosting',
             fontsize=13, fontweight='bold', pad=12)
ax.grid(axis='y', visible=False)
ax.set_axisbelow(True)
sns.despine(ax=ax, left=True)
ax.tick_params(axis='y', length=0)
plt.tight_layout()
fig.savefig(FIGURES / '07b_feature_importances.png', bbox_inches='tight')
fig.savefig(FIGURES / '07b_feature_importances.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ── 4c: Confusion Matrix for the better model ──
if gbc_auc >= lr_auc:
    better_name, y_pred_cm = 'Gradient Boosting', gbc.predict(X_test)
else:
    better_name, y_pred_cm = 'Logistic Regression', lr.predict(X_test_sc)

cm     = confusion_matrix(y_test, y_pred_cm, normalize='true')
cm_raw = confusion_matrix(y_test, y_pred_cm)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=False, cmap='Blues', vmin=0, vmax=1,
            linewidths=0.5, linecolor='white', ax=ax,
            xticklabels=['Enrolled', 'Dropped out'],
            yticklabels=['Enrolled', 'Dropped out'])
for i in range(2):
    for j in range(2):
        ax.text(j + 0.5, i + 0.42, f'{cm[i,j]:.1%}',
                ha='center', va='center', fontsize=13, fontweight='bold',
                color='white' if cm[i, j] > 0.5 else '#222222')
        ax.text(j + 0.5, i + 0.62, f'n = {cm_raw[i,j]:,}',
                ha='center', va='center', fontsize=9,
                color='white' if cm[i, j] > 0.5 else '#555555')
ax.set_xlabel('Predicted label', labelpad=8)
ax.set_ylabel('True label', labelpad=8)
ax.set_title(f'Confusion Matrix — {better_name}\n(row-normalised)',
             fontsize=12, fontweight='bold', pad=12)
plt.tight_layout()
fig.savefig(FIGURES / '07c_confusion_matrix.png', bbox_inches='tight')
fig.savefig(FIGURES / '07c_confusion_matrix.pdf', bbox_inches='tight')
plt.show()

## Step 5 — Provincial Analysis: Actionable Household Predictors

The national model averages over provinces with structurally different dropout drivers. Four province-specific logistic regressions are trained (80/20 stratified split each), with AUC evaluated on the **held-out test set**.

**Chart selection rule**: `school_stage_middle` and `school_stage_secondary` are *excluded* from the top-5 feature ranking for the provincial chart. Stage-level effects are structural (dropout rates mechanically rise at transitions) and the same everywhere — showing them would crowd out the household and socioeconomic predictors that provincial policymakers can actually target. The chart is titled accordingly: *actionable* predictors.

**Reading coefficients:**
- **Positive** → that feature *increases* predicted dropout probability
- **Negative** → that feature *reduces* predicted dropout probability
- Magnitudes are standardised, so bars are directly comparable within and across provinces

In [ ]:
PROVINCES_P5 = ['Punjab', 'Sindh', 'KPK', 'Balochistan']
PROV_COLORS  = {'Punjab': '#1f77b4', 'Sindh': '#d62728',
                'KPK':    '#2ca02c', 'Balochistan': '#e07b39'}
STAGE_FEATS  = {'school_stage_middle', 'school_stage_secondary'}

prov_coefs   = {}
prov_results = {}

for prov in PROVINCES_P5:
    sub = df[df['province'] == prov]
    Xp  = sub[FEATURE_COLS].astype(float)
    yp  = sub['target']

    if yp.nunique() < 2 or yp.sum() < 20:
        print(f'{prov}: skipped (insufficient dropout events)')
        continue

    Xp_tr, Xp_te, yp_tr, yp_te = train_test_split(
        Xp, yp, test_size=0.20, random_state=RANDOM_STATE, stratify=yp
    )
    sc_p     = StandardScaler()
    Xp_tr_sc = sc_p.fit_transform(Xp_tr)
    Xp_te_sc = sc_p.transform(Xp_te)

    lr_p = LogisticRegression(
        class_weight='balanced', max_iter=1000, C=0.5, random_state=RANDOM_STATE
    )
    lr_p.fit(Xp_tr_sc, yp_tr)

    prob_te = lr_p.predict_proba(Xp_te_sc)[:, 1]
    fp, tp, _ = roc_curve(yp_te, prob_te)
    test_auc  = auc(fp, tp)

    prov_coefs[prov]   = pd.Series(lr_p.coef_[0], index=FEATURE_COLS)
    prov_results[prov] = dict(
        n_train=len(yp_tr), n_test=len(yp_te),
        dropout_train=int(yp_tr.sum()), dropout_test=int(yp_te.sum()),
        test_auc=test_auc
    )
    print(f'{prov}: n_train={len(yp_tr):,}, dropouts_train={yp_tr.sum()}, '
          f'n_test={len(yp_te):,}, dropouts_test={yp_te.sum()}, '
          f'test_AUC={test_auc:.3f}')

# Rename raw feature names to pretty labels
coef_df = pd.DataFrame(prov_coefs).rename(index=PRETTY)

# Exclude school_stage dummies from ranking — select actionable household predictors only
stage_pretty = {PRETTY[f] for f in STAGE_FEATS if f in PRETTY}
household_coef = coef_df[~coef_df.index.isin(stage_pretty)]

top5_feats = household_coef.abs().mean(axis=1).nlargest(5).index.tolist()

print('\nTop 5 actionable features by mean |coef| (school_stage excluded):')
print(coef_df.loc[top5_feats].round(3))

print('\n── School-stage coefficients (for reference) ──')
print(coef_df.loc[list(stage_pretty)].round(3))

In [ ]:
# ── 5b: Provincial coefficient chart ──
n_feat   = len(top5_feats)
n_prov   = len(prov_coefs)
x        = np.arange(n_feat)
w        = 0.18
offsets  = np.linspace(-(n_prov - 1) / 2, (n_prov - 1) / 2, n_prov) * w

fig, ax = plt.subplots(figsize=(12, 6))

for (prov, coefs), offset in zip(prov_coefs.items(), offsets):
    pretty_coefs = coefs.rename(index=PRETTY)
    vals = [pretty_coefs.get(f, np.nan) for f in top5_feats]
    ax.bar(x + offset, vals, width=w, color=PROV_COLORS[prov],
           label=prov, zorder=3, edgecolor='white', linewidth=0.4)

ax.axhline(0, color='#444444', linewidth=0.9, zorder=2)
ax.set_xticks(x)
ax.set_xticklabels(top5_feats, fontsize=10)
ax.set_ylabel('Standardised LR coefficient\n'
              '(positive = ↑ dropout risk,  negative = ↓ dropout risk)', labelpad=8)
ax.set_title('Top 5 Actionable Dropout Predictors by Province\n'
             '(Logistic Regression, balanced weights, standardised features, test-set AUC)',
             fontsize=12, fontweight='bold', pad=12)
ax.legend(frameon=False, fontsize=10)
ax.grid(axis='x', visible=False)
ax.set_axisbelow(True)
sns.despine(ax=ax)
plt.tight_layout()

fig.savefig(FIGURES / '07d_provincial_coefficients.png', bbox_inches='tight')
fig.savefig(FIGURES / '07d_provincial_coefficients.pdf', bbox_inches='tight')
plt.show()

## Step 6 — Multicollinearity Check: Variance Inflation Factors

The Variance Inflation Factor (VIF) for feature *j* measures how much its variance is inflated by collinearity with other features. The rule of thumb: **VIF < 5 = acceptable**, VIF 5–10 = moderate concern, VIF > 10 = serious problem.

This check validates that replacing the continuous `age`/`grade` pair with the categorical `school_stage` dummies has resolved the collinearity that made the previous notebook's provincial chart uninterpretable.

In [ ]:
X_full = df[FEATURE_COLS].astype(float)

vif_df = pd.DataFrame({
    'Feature': FEATURE_COLS,
    'VIF': [variance_inflation_factor(X_full.values, i)
            for i in range(X_full.shape[1])],
})
vif_df['Pretty name'] = vif_df['Feature'].map(PRETTY)
vif_df = vif_df[['Pretty name', 'VIF']].sort_values('VIF', ascending=False)
vif_df['Status'] = vif_df['VIF'].apply(
    lambda v: '⚠ borderline' if 4 <= v < 5 else ('✗ HIGH' if v >= 5 else '✓ OK')
)

print('Variance Inflation Factors — full feature matrix')
print('=' * 55)
print(vif_df.to_string(index=False, float_format='{:.3f}'.format))
print('=' * 55)
print(f'Max VIF: {vif_df.VIF.max():.3f}   Threshold: 5.0')

assert vif_df['VIF'].max() < 5, 'At least one feature exceeds VIF threshold of 5'
print('\nAll features below VIF threshold of 5.0 ✓')

# Note on borderline feature
border = vif_df[vif_df['Status'] == '⚠ borderline']
if len(border):
    for _, row in border.iterrows():
        print(f"\nNote: '{row['Pretty name']}' (VIF={row['VIF']:.3f}) is borderline. "
              f"This reflects natural correlation among household assets "
              f"(wealthier households tend to have electricity AND smartphones). "
              f"The VIF remains under 5, so L2 regularisation in the logistic "
              f"regression provides sufficient control.")

## Policy Interpretation

### What changed from the previous version

The previous notebook included raw `age` and `grade` as continuous features (r = 0.907). In standardised logistic regression, collinear predictors produce inflated, unstable coefficients — the model assigned values of ±9–12 to age and grade, which dominated every chart and obscured every other predictor. The fix replaces both with a three-level categorical `school_stage` (primary/middle/secondary), which captures the policy-relevant question — *at which transition point do dropouts concentrate?* — without numerical collinearity. VIF confirms all 14 features are now below the 5.0 threshold (Step 6).

The drop from the old AUC (~0.96) to **0.852** is expected and honest: the old model's high AUC was largely driven by age and grade, which are near-perfect proxies for school stage and trivially predict dropout. The new model uses only features a policymaker can observe and act on.

---

### Provincial test-set performance

| Province    | Test AUC | Dropout events (test set) |
|-------------|----------|--------------------------|
| Balochistan | 0.855    | 480                      |
| Sindh       | 0.774    | 102                      |
| Punjab      | 0.733    | 91                       |
| KPK         | 0.710    | 89                       |

---

### Reading the provincial coefficient chart (Figure 07d)

The chart shows **household and socioeconomic predictors only** — school_stage dummies are excluded because their direction is structurally fixed (dropout rates always rise at transitions) and identical across provinces, making them uninformative for provincial targeting.

**Key features and their policy meaning:**

| Feature | Expected direction | Policy lever |
|---|---|---|
| **Education expense** | Negative (↓ risk) | The strongest single household predictor. Zero education expense is the clearest early signal of disengagement before formal dropout. Cost removal — fee waivers, school supply grants — is the most direct lever, especially in Balochistan (coef = −0.82) |
| **Receives Ehsaas** | Negative (↓ risk) in Sindh/KPK | Ehsaas is a conditional cash transfer requiring school enrolment. Its negative coefficient confirms the programme is reaching at-risk households and conditionality is working. Expanding coverage in high-dropout districts is actionable |
| **Earning members** | Negative (↓ risk) | More adult earners → less pressure on child labour contribution. Household income diversification reduces dropout pressure consistently across all provinces |
| **Gender: Male** | Negative (↓ risk) | Girls face higher dropout risk than boys across all four provinces, consistent with safety, distance, and social-norms barriers. The effect is largest in KPK |
| **Receives BISP** | Positive (↑ risk) | BISP targets the poorest households. The positive coefficient reflects that BISP receipt is a proxy for deep poverty — the transfers exist *because* the household is at high risk. Unlike Ehsaas, BISP is unconditional, so school attendance is not required to receive it |

**The BISP–Ehsaas contrast is the most actionable finding in this chart.** Ehsaas (conditional) *reduces* dropout risk in Sindh and KPK; BISP (unconditional) is *associated with higher risk* because it proxies extreme poverty. This does not mean BISP harms schooling — it means the programme's poverty-targeting is so sharp that BISP receipt is a near-synonym for the conditions that cause dropout. The policy implication: expand Ehsaas coverage in high-BISP districts, or attach school-attendance conditionality to BISP transfers in the highest-dropout areas.

**Provincial heterogeneity is the main policy message of this chart.** Education expense is −0.82 in Balochistan but only −0.17 in Sindh. A uniform national programme applying Punjab-calibrated weights will systematically under-serve Balochistan. Province-specific model variants are warranted.

---

### Model limitations

1. **Grade missing (~45% of dropouts)**: children who dropped out without a recorded grade are excluded from the analytic sample. The models are calibrated on mid-year dropouts and will underestimate risk for children who never formally entered or left at the very beginning of a school year.
2. **Cross-sectional data**: the model predicts *observed* dropout status at survey time, not future dropout risk. Applying it prospectively requires that the relationship between household predictors and dropout remains stable across years.
3. **Household self-report**: BISP receipt, asset ownership, and flood exposure are self-reported and may be under- or over-stated.

In [ ]:
print("=== Dropout count diagnostics ===")

# 1. Raw count of all dropouts in the source file (before any row filtering)
#    'raw' is loaded from child_merged.csv with LOAD_COLS;
#    enrollment_status values include 'Enrolled', 'Dropped out', and others
#    (e.g. never-enrolled children) that are excluded when building 'base'
raw_dropouts = (raw['enrollment_status'] == 'Dropped out').sum()
print(f"Raw dropouts in source file: {raw_dropouts}")

# 2. Count after restricting to Enrolled/Dropped-out AND requiring non-missing grade
#    'base' already excludes never-enrolled; filter further on grade.notna()
with_grade = (base[base['grade'].notna()]['enrollment_status'] == 'Dropped out').sum()
print(f"Dropouts with non-missing grade: {with_grade}")

# 3. Count after requiring both non-missing grade AND non-missing gender
#    Matches the dropna(subset=['grade','gender']) step that builds df,
#    but before the school_stage cut removes out-of-range grades (0, 11-14)
both_present = base[base['grade'].notna() & base['gender'].notna()]
model_dropouts_pre_stage = (both_present['enrollment_status'] == 'Dropped out').sum()
print(f"Dropouts with non-missing grade and gender (pre school_stage filter): {model_dropouts_pre_stage}")

# 4. Final model sample: df has also had invalid grades removed via school_stage cut
model_dropouts = int(df['target'].sum())
print(f"Dropouts in main model sample (after school_stage filter): {model_dropouts}")

# 5. Total sample and rate
print(f"Total children in main model sample: {len(df)}")
print(f"Dropout rate: {model_dropouts / len(df):.4f}")

In [ ]:
# Compute precision exactly from TP/FP pairs reported in the paper
rows = [
    (0.05, 770, 26252),
    (0.10, 753, 22881),
    (0.20, 705, 16725),
    (0.30, 657, 12069),
    (0.50, 574, 5356),
]
print(f"{'Threshold':>10} {'TP':>6} {'FP':>7} {'Precision (computed)':>22}")
for thr, tp, fp in rows:
    prec = tp / (tp + fp)
    print(f"{thr:>10.2f} {tp:>6} {fp:>7} {prec:>22.4f}")

print()
print("Paper-reported precisions: 0.029, 0.032, 0.040, 0.052, 0.097")